# VascuQuest JAX split-solver qualification

This notebook qualifies the structure-preserving accelerated Virtual Disease solver on **one deterministic canonical PWDB subject across all four frozen disease models**. It is a numerical/software qualification, not clinical validation.

The accelerated scheme is `jax-exact-loss-rkc2-voigt-ssprk2-v1`: exact Young–Seeley focal-loss propagation, globally coupled RKC2 stabilization of the PWDB Voigt source, and the frozen hyperbolic/network operator advanced with SSP-RK2. NumPy remains the default/reference backend.

After the four-disease gate passes, the same carotid-stenosis case is rerun at progressively halved wave-CFL values to require approximately second-order temporal self-convergence.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess, sys, json

REPO = Path('/content/VascuQuest-jax-split-qualification')
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run([
    'git', 'clone', '--depth', '1', '--branch',
    'release/parameterized-cohort-qualification',
    'https://github.com/KNOWDYN/VascuQuest.git', str(REPO)
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
CODE_REVISION = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()

import jax
jax.config.update('jax_enable_x64', True)
devices = jax.devices()
gpu_devices = [d for d in devices if d.platform == 'gpu']
print('Code revision:', CODE_REVISION)
print('JAX version:', jax.__version__)
print('JAX devices:', devices)
if not gpu_devices:
    raise RuntimeError('No JAX GPU device is available. Select a Colab GPU runtime and restart.')
print('JAX GPU gate: PASS ->', gpu_devices)


In [ ]:
# Stage canonical PWDB artifacts to local SSD. No recursive Drive search.
DRIVE_PWDB_SOURCE = Path('/content/drive/MyDrive/VQ_WallWork_CBM/source/PWDB_3275625')
LOCAL_SOURCE = Path('/content/vascuquest-pwdb-source')
OUTPUT_ROOT = Path('/content/drive/MyDrive/VascuQuest/jax_split_one_subject_qualification') / CODE_REVISION[:12]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
STAGE_REPORT = OUTPUT_ROOT / 'source-stage.json'
REPORT = OUTPUT_ROOT / 'jax-split-one-subject-qualification.json'

stage_cmd = [
    sys.executable, str(REPO / 'tests/full_data/parameterized_cohort_colab_stage.py'),
    '--drive-source-dir', str(DRIVE_PWDB_SOURCE),
    '--local-source', str(LOCAL_SOURCE),
    '--report', str(STAGE_REPORT),
]
print('$', ' '.join(stage_cmd), flush=True)
subprocess.run(stage_cmd, check=True, cwd=REPO)
print(STAGE_REPORT.read_text())
print('PWDB local-SSD source gate: PASS')


In [ ]:
# Confirm the package surface before spending GPU time.
from vascuquest.disease.solver import create_disease_solver
from vascuquest.disease.solver.disease_finite_volume import DiseaseOneDSolver
from vascuquest.disease.solver.jax_split_disease import JAX_SPLIT_SCHEME_ID, JaxDiseaseOneDSolver
numpy_solver = create_disease_solver()
accelerated_solver = create_disease_solver(backend='jax')
assert isinstance(numpy_solver, DiseaseOneDSolver)
assert isinstance(accelerated_solver, JaxDiseaseOneDSolver)
print('NumPy frozen default: PASS')
print('Accelerated scheme:', JAX_SPLIT_SCHEME_ID)


In [ ]:
# Run the decisive one-subject/four-disease numerical qualification.
cmd = [
    sys.executable, str(REPO / 'tests/full_data/jax_split_one_subject_qualification.py'),
    '--source', str(LOCAL_SOURCE),
    '--report', str(REPORT),
    '--code-revision', CODE_REVISION,
]
print('$', ' '.join(cmd), flush=True)
completed = subprocess.run(cmd, cwd=REPO)
if completed.returncode != 0:
    if REPORT.exists():
        print('Persisted failure/partial record:')
        print(REPORT.read_text())
    raise RuntimeError(f'JAX split qualification failed with exit code {completed.returncode}')
print('Four-disease qualification: PASS')


In [ ]:
# Require approximately second-order temporal self-convergence.
refine_cmd = [
    sys.executable, str(REPO / 'tests/full_data/jax_split_temporal_refinement.py'),
    '--source', str(LOCAL_SOURCE),
    '--report', str(REPORT),
]
print('$', ' '.join(refine_cmd), flush=True)
refined = subprocess.run(refine_cmd, cwd=REPO)
if refined.returncode != 0:
    if REPORT.exists():
        print('Qualification record after temporal-refinement failure:')
        print(REPORT.read_text())
    raise RuntimeError(f'Temporal-refinement qualification failed with exit code {refined.returncode}')
print('Temporal-order gate: PASS')


In [ ]:
payload = json.loads(REPORT.read_text())
assert payload['status'] == 'PASS'
assert payload.get('temporal_refinement', {}).get('status') == 'PASS'
print(json.dumps({
    'status': payload['status'],
    'code_revision': payload['code_revision'],
    'canonical_subject_id': payload['canonical_subject_id'],
    'source_age_years': payload['source_age_years'],
    'numerical_scheme_id': payload['numerical_scheme_id'],
    'full_numpy_anchor': payload['full_numpy_anchor'],
    'temporal_refinement': payload['temporal_refinement'],
}, indent=2, sort_keys=True))
print('\nPer-disease limiter attribution:')
for case in payload['cases']:
    info = case['accelerated_full_solve']['limiter_attribution']
    print(case['condition'], json.dumps(info, sort_keys=True))
print('\nDurable evidence:', REPORT)
